# Factor Analysis

This notebook will examine some of the variables that are going to be used as signals. Before generating signals for these portfolios they should be tested amongst the tradable securities to examine their factor exposure. Therefore this notebook will examine how the different ways the signals influence the returns. 

In [1]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import statsmodels.api as sm

from   matplotlib import cm
import matplotlib.pyplot as plt
from   matplotlib.ticker import FuncFormatter

In [2]:
note_path = os.getcwd()
repo_path = os.path.abspath(os.path.join(note_path, ".."))
src_path  = os.path.join(repo_path, "src")
data_path = os.path.join(repo_path, "data")
prep_path = os.path.join(data_path, "PrepData")
fact_path = os.path.join(data_path, "FactorAnalysis")
pic_path  = os.path.join(repo_path, "writeup", "pictures")

In [3]:
sys.path.append(src_path)
from FactorExposureTools import FactorExposureTools

In [4]:
path      = os.path.join(data_path, "FXTickerGuide.xlsx")
df_ticker = (pd
    .read_excel(io = path, sheet_name = "TickerGuide")
    .rename(columns = {
        "ticker"       : "fx_ticker",
        "etf_benchmark": "index_ticker"})
    .drop(columns = ["Active", "FACTOR_ACTIVE"])
    .assign(etf_ticker = lambda x: x.etf_ticker.str.split(" ").str[0])
    .assign(fx_name = lambda x: x.shorthand_name + " " + x.return_type.str.capitalize()))

In [5]:
FX_RENAMER = (df_ticker
    .set_index("fx_ticker")
    .fx_name
    .to_dict())

vol_renamer = {
        "lagged": "Lag",
        "none"  : "None",
        "perfect": "Perf"}

signal_renamer = {
    "no_lag"  : "No Lag", 
    "with_lag": "Lagged"}

param_renamer = {
    "param_val" : "Coef",
    "pvalue"    : r"$p$-value",
    "tvalue"    : r"$t$-value",
    "r2"        : r"$R^2$"}

group_renamer = {
    'em_carry'       : "EM Carry", 
    'g10_carry'      : "G10 Carry", 
    'liquid_future'  : "Liquid Futures", 
    'illiquid_future': "Illiquid Futures"}

RENAMER = {**FX_RENAMER, **vol_renamer, **param_renamer, **group_renamer, **signal_renamer}

In [6]:
fx_country_mapper = (df_ticker[
    ["fx_ticker", "country"]]
    .set_index("fx_ticker")
    .country
    .to_dict())

FLOAT_FORMAT = lambda x: f"{x:.2f}"

# Comparing Sovereign Equity Returns with FX Returns

In this section start by comparing the sovereign equity returns with the FX returns. The idea is to start with the raw equity returns to set a baseline model and then compare the equity returns differential. The overall interest is in the following
1. Testing to see if the factors are statistically significant
2. Examining the relationship between the equity returns and their FX counterparts

In [7]:
path = os.path.join(fact_path, "ReturnsOLSModels.pkl")
with open(path, "rb") as f: rtn_models = pickle.load(f)

In [8]:
df_rtn_params = FactorExposureTools().get_ols_params(rtn_models)

In [9]:
df_rtn_param_prep = (df_rtn_params
    .assign(
        str_string = lambda x: x.group_var.str.split("-"),
        fx_ticker  = lambda x: x.str_string.str[0],
        vol_target = lambda x: x.str_string.str[1],
        asset_type = lambda x: x.str_string.str[2],
        etf_ticker = lambda x: x.str_string.str[3])
    .drop(columns = ["str_string", "group_var"]))

df_rtn_beta = (df_rtn_param_prep
    .loc[lambda x: x.param_name == "etf_rtn"]
    .drop(columns = ["param_name"])
    .melt(id_vars = ["etf_ticker", "fx_ticker", "vol_target", "asset_type"]))

Below is a table that has the regression results between all FX returns and the equity returns ran instantaneously

In [10]:
(df_rtn_beta
    .assign(country = lambda x: x.fx_ticker.map(fx_country_mapper))
    .loc[lambda x: x.vol_target.isin(["perfect", "none"])]
    .replace(RENAMER)
    .rename(columns = {
        "etf_ticker": "ETF",
        "fx_ticker" : "FX",
        "variable"  : "",
        "vol_target": "Vol. Target",
        "country"   : "Country"})
    .pivot(index = ["ETF", "Country", "FX"], columns = ["", "Vol. Target"], values = "value"))

Coef                $p$-value  \
Vol. Target                       None      Perf           None   
ETF  Country      FX                                              
ARGT Argentina    ARS Carry   0.082595  0.035126   1.336358e-05   
COLO Colombia     COP Carry   0.321565  0.234087   0.000000e+00   
ECH  Chile        CLP Carry   0.269471  0.208859   0.000000e+00   
EDEN Denmark      DKK Carry   0.145107  0.182225  2.121517e-104   
EIDO Indonesia    IDR Carry   0.072750  0.111620   4.722098e-98   
EIS  Israel       ISR Future  0.198298  0.210841  1.791064e-298   
ENOR Norway       NOK Carry   0.335389  0.270250   0.000000e+00   
                  NOK Future  0.326852  0.262355   0.000000e+00   
ENZL New Zealand  NZD Carry   0.361125  0.320415   0.000000e+00   
                  NZD Future  0.354008  0.304500   0.000000e+00   
EPHE Philippines  PHP Carry   0.051285  0.103651   3.379634e-47   
EPOL Poland       PLN Carry   0.291538  0.223487   0.000000e+00   
                  PLN Future  0.286513  0.217567   0.000000e+00   
EPU  Peru         PEN Carry   0.094871  0.167819  1.798465e-162   
EWA  Australia    AUD Carry   0.303772  0.219035   0.000000e+00   
                  AUD Future  0.277878  0.200562   0.000000e+00   
EWC  Canada       CAD Carry   0.225581  0.247734   0.000000e+00   
                  CAD Future  0.205662  0.227411   0.000000e+00   
EWD  Sweden       SEK Carry   0.198889  0.149534   0.000000e+00   
                  SEK Future  0.210591  0.155929   0.000000e+00   
EWH  Hong Kong    HKD Carry   0.001376  0.022674   5.577832e-09   
EWJ  Japan        JPY Carry   0.027247  0.023603   2.024639e-07   
                  JPY Future  0.040858  0.029081   1.578510e-14   
EWL  Switzerland  CHF Carry   0.126456  0.112455  2.510163e-110   
                  CHF Future  0.128382  0.112095  1.740757e-113   
EWM  Malaysia     MYR Carry   0.055701  0.090175   2.226136e-94   
EWS  Singapore    SGD Carry   0.086144  0.157042  2.315662e-297   
EWT  Taiwan       TWD Carry   0.034209  0.082091   3.609574e-84   
EWU  UK           GBP Carry   0.163831  0.158205  2.119276e-305   
                  GBP Future  0.155778  0.148644  1.467623e-267   
EWW  Mexico       MXN Carry   0.272863  0.218379   0.000000e+00   
                  MXN Future  0.243375  0.195674   0.000000e+00   
EWY  Korea        KRW Carry   0.112106  0.100103  2.113670e-204   
                  KRW Future  0.191638  0.161531   0.000000e+00   
EWZ  Brazil       BRL Carry   0.309207  0.177898   0.000000e+00   
                  BRL Future  0.284512  0.167689   0.000000e+00   
EZA  South Africa ZAR Carry   0.370722  0.209229   0.000000e+00   
EZU  Euro         EUR Carry   0.151981  0.148592  8.067524e-257   
                  EUR Future  0.144419  0.141307  5.186726e-231   
GLCR Iceland      ISK Carry   0.155909  0.198228   2.041979e-08   
INDA India        INR Carry   0.109518  0.169941  1.339227e-137   
                  INR Future  0.165117  0.244429  1.520319e-296   
MCHI China        CNY Carry   0.034573  0.087240   1.002421e-62   
                  CNY Future -0.036809 -0.080559   1.729222e-39   
THD  Thailand     THB Carry   0.063793  0.113606   1.303832e-88   
TUR  Turkey       TRY Carry   0.278913  0.159445   0.000000e+00   

                                             $t$-value                $R^2$  \
Vol. Target                            Perf       None       Perf      None   
ETF  Country      FX                                                          
ARGT Argentina    ARS Carry    2.506035e-11   4.359806   6.693111  0.004992   
COLO Colombia     COP Carry    0.000000e+00  52.215390  47.094195  0.387530   
ECH  Chile        CLP Carry    0.000000e+00  49.665744  46.037941  0.348319   
EDEN Denmark      DKK Carry   1.922168e-100  22.437287  21.971066  0.123832   
EIDO Indonesia    IDR Carry    2.766092e-87  21.611297  20.301013  0.104648   
EIS  Israel       ISR Future  4.919001e-271  39.935086  37.760187  0.263178   
ENOR Norway       NOK Carry    0.000000

In [11]:
fx_ticker_map = (df_ticker
    .set_index("fx_ticker")
    .group
    .to_dict())

In [12]:
df_compare = (df_rtn_beta
    .loc[lambda x: x.variable.isin(["param_val", "tvalue"])]
    .loc[lambda x: x.vol_target != "lagged"]
    .assign(group = lambda x: x.fx_ticker.map(fx_ticker_map))
    .drop(columns = ["etf_ticker", "asset_type"])
    .pivot(index = ["fx_ticker", "variable", "group"], columns = "vol_target", values = "value")
    .assign(improved = lambda x: np.where(x.perfect > x.none, "with_vol", "with_none")))

In [13]:
df_all_compare = (df_compare
    .reset_index()
    [["improved", "variable", "fx_ticker"]]
    .groupby(["improved", "variable"])
    .agg("count")
    .assign(group = "all")
    .reset_index())

df_group_compare = (df_compare
    .reset_index()
    [["variable", "fx_ticker", "group", "improved"]]
    .groupby(["improved", "variable", "group"])
    .agg("count")
    .reset_index())

In [14]:
display(pd
    .concat([df_all_compare, df_group_compare])
    .pivot(index = ["variable", "group"], columns = "improved", values = "fx_ticker")
    .assign(cuml_val = lambda x: x.sum(axis = 1))
    .reset_index()
    .rename(columns = {"variable": "param"})
    .melt(id_vars = ["param", "group", "cuml_val"])
    .assign(ratio = lambda x: x.value / x.cuml_val)
    .replace(RENAMER)
    .replace({
        "all"      : "All",
        "with_none": "No Vol Target",
        "with_vol" : "With Vol Target"})
    .rename(columns = {
        "improved": "Improved",
        "param"   : "Param"})
    .pivot(index = "group", columns = ["Improved", "Param"], values = "ratio")
    .fillna(0))

Improved         No Vol Target           With Vol Target          
Param                     Coef $t$-value            Coef $t$-value
group                                                             
All                   0.652174  0.891304        0.347826  0.108696
EM Carry              0.478261  0.826087        0.521739  0.173913
G10 Carry             0.857143  1.000000        0.142857  0.000000
Illiquid Futures      0.714286  0.857143        0.285714  0.142857
Liquid Futures        0.888889  1.000000        0.111111  0.000000

In [15]:
path = os.path.join(fact_path, "ReturnsDifferentialOLSModels.pkl")
with open(path, "rb") as f: spread_models = pickle.load(f)

In [16]:
df_spread_params = FactorExposureTools().get_ols_params(spread_models)

In [17]:
df_spread_param_prep = (df_spread_params
    .loc[lambda x: x.param_name == "spread"]
    .assign(
        str_string = lambda x: x.group_var.str.split("-"),
        fx_ticker  = lambda x: x.str_string.str[0],
        vol_target = lambda x: x.str_string.str[1],
        asset_type = lambda x: x.str_string.str[2],
        etf_ticker = lambda x: x.str_string.str[3])
    .drop(columns = ["str_string", "group_var", "param_name"]))

Then compare the regression results for using the returns spread. 

In [18]:
(df_spread_param_prep
    .loc[lambda x: x.vol_target.isin(["none", "perfect"])]
    .melt(id_vars = ["fx_ticker", "vol_target", "asset_type", "etf_ticker"])
    .assign(country = lambda x: x.fx_ticker.map(fx_country_mapper))
    .replace(RENAMER)
    .rename(columns = {
        "etf_ticker": "ETF",
        "country"   : "Country",
        "fx_ticker" : "FX",
        "vol_target": "Vol. Target",
        "variable"  : ""})
    .pivot(index = ["ETF", "Country", "FX"], columns = ["", "Vol. Target"], values = "value"))

Coef                $p$-value  \
Vol. Target                       None      Perf           None   
ETF  Country      FX                                              
ARGT Argentina    ARS Carry   0.125669  0.046794   1.122526e-07   
COLO Colombia     COP Carry   0.272432  0.196779  4.859221e-223   
ECH  Chile        CLP Carry   0.257177  0.201783  3.557453e-243   
EDEN Denmark      DKK Carry   0.169822  0.216376   1.142702e-91   
EIDO Indonesia    IDR Carry   0.085922  0.128594   7.956132e-89   
EIS  Israel       ISR Future  0.227051  0.240259  2.111530e-156   
ENOR Norway       NOK Carry   0.372797  0.312518  8.812551e-304   
                  NOK Future  0.381957  0.316401  1.336942e-314   
ENZL New Zealand  NZD Carry   0.279014  0.260741  2.730424e-162   
                  NZD Future  0.297142  0.269972  7.480320e-174   
EPHE Philippines  PHP Carry   0.054049  0.107686   7.054277e-38   
EPOL Poland       PLN Carry   0.358221  0.287994   0.000000e+00   
                  PLN Future  0.358882  0.284120   0.000000e+00   
EPU  Peru         PEN Carry   0.072596  0.119062   1.549155e-67   
EWA  Australia    AUD Carry   0.306839  0.240772   0.000000e+00   
                  AUD Future  0.303622  0.235651   0.000000e+00   
EWC  Canada       CAD Carry   0.169995  0.196710  3.981306e-205   
                  CAD Future  0.191609  0.219766  1.428241e-266   
EWD  Sweden       SEK Carry   0.266320  0.213589   0.000000e+00   
                  SEK Future  0.348897  0.271101   0.000000e+00   
EWH  Hong Kong    HKD Carry   0.000898  0.013296   3.313747e-03   
EWJ  Japan        JPY Carry   0.177708  0.154306  2.811273e-168   
                  JPY Future  0.182457  0.150560  4.253693e-172   
EWL  Switzerland  CHF Carry   0.256827  0.238166  4.697531e-306   
                  CHF Future  0.250482  0.231141  7.410192e-289   
EWM  Malaysia     MYR Carry   0.041133  0.067457   9.575077e-45   
EWS  Singapore    SGD Carry   0.072930  0.144225  2.571597e-112   
EWT  Taiwan       TWD Carry   0.045385  0.108690   4.784683e-86   
EWU  UK           GBP Carry   0.236468  0.248816  4.859008e-290   
                  GBP Future  0.232764  0.241517  4.535455e-274   
EWW  Mexico       MXN Carry   0.261883  0.215112   0.000000e+00   
                  MXN Future  0.246884  0.200417   0.000000e+00   
EWY  Korea        KRW Carry   0.166704  0.151936  2.512809e-253   
                  KRW Future  0.282880  0.239014   0.000000e+00   
EWZ  Brazil       BRL Carry   0.365392  0.212486   0.000000e+00   
                  BRL Future  0.343103  0.204671   0.000000e+00   
EZA  South Africa ZAR Carry   0.464624  0.271511   0.000000e+00   
EZU  Euro         EUR Carry   0.346810  0.357000   0.000000e+00   
                  EUR Future  0.339314  0.349118   0.000000e+00   
GLCR Iceland      ISK Carry   0.155230  0.194500   2.667198e-07   
INDA India        INR Carry   0.132569  0.201833  1.608691e-131   
                  INR Future  0.163848  0.227515  4.254168e-175   
MCHI China        CNY Carry   0.035008  0.093347   9.839679e-45   
                  CNY Future -0.039826 -0.089174   1.724941e-33   
THD  Thailand     THB Carry   0.083792  0.145271   3.142219e-89   
TUR  Turkey       TRY Carry   0.299905  0.167264   0.000000e+00   

                                             $t$-value                $R^2$  \
Vol. Target                            Perf       None       Perf      None   
ETF  Country      FX                                                          
ARGT Argentina    ARS Carry    1.094159e-12   5.315903   7.142448  0.007403   
COLO Colombia     COP Carry   1.419562e-189  33.859198  30.902335  0.210147   
ECH  Chile        CLP Carry   3.453310e-223  35.409219  33.734513  0.213640   
EDEN Denmark      DKK Carry    6.390303e-91  20.909392  20.817012  0.109322   
EIDO Indonesia    IDR Carry    3.152880e-75  20.492650  18.747311  0.095098   
EIS  Israel       ISR Future  1.308783e-141  27.740417  26.275138  0.147010   
ENOR Norway       NOK Carry   7.022007e

In [19]:
df_compare = (df_spread_params
    .loc[lambda x: x.param_name == "spread"]
    .assign(
        str_split  = lambda x: x.group_var.str.split("-"),
        fx_ticker  = lambda x: x.str_split.str[0],
        vol_target = lambda x: x.str_split.str[1])
    [["param_val", "tvalue", "fx_ticker", "vol_target"]]
    .melt(id_vars = ["fx_ticker", "vol_target"])
    .loc[lambda x: x.vol_target != "lagged"]
    .assign(group = lambda x: x.fx_ticker.map(fx_ticker_map))
    .pivot(index = ["fx_ticker", "variable", "group"], columns = "vol_target", values = "value")
    .assign(improved = lambda x: np.where(x.perfect > x.none, "with_vol", "with_none")))

In [20]:
df_all_compare = (df_compare
    .reset_index()
    [["improved", "variable", "fx_ticker"]]
    .groupby(["improved", "variable"])
    .agg("count")
    .assign(group = "all")
    .reset_index())

df_group_compare = (df_compare
    .reset_index()
    [["variable", "fx_ticker", "group", "improved"]]
    .groupby(["improved", "variable", "group"])
    .agg("count")
    .reset_index())

In [21]:
display(pd
    .concat([df_all_compare, df_group_compare])
    .pivot(index = ["variable", "group"], columns = "improved", values = "fx_ticker")
    .assign(cuml_val = lambda x: x.sum(axis = 1))
    .reset_index()
    .rename(columns = {"variable": "param"})
    .melt(id_vars = ["param", "group", "cuml_val"])
    .assign(ratio = lambda x: x.value / x.cuml_val)
    .replace(RENAMER)
    .replace({
        "all"      : "All",
        "with_none": "No Vol Target",
        "with_vol" : "With Vol Target"})
    .rename(columns = {
        "improved": "Improved",
        "param"   : "Param"})
    .pivot(index = "group", columns = ["Improved", "Param"], values = "ratio")
    .fillna(0))

Improved         No Vol Target           With Vol Target          
Param                     Coef $t$-value            Coef $t$-value
group                                                             
All                   0.565217  0.826087        0.434783  0.173913
EM Carry              0.478261  0.782609        0.521739  0.217391
G10 Carry             0.571429  0.857143        0.428571  0.142857
Illiquid Futures      0.714286  0.857143        0.285714  0.142857
Liquid Futures        0.666667  0.888889        0.333333  0.111111

Now compare the average $\beta$ and $t$-stat per each group

In [22]:
path = os.path.join(fact_path, r"ReturnsDifferentialTrendOLSModels.pkl")
with open(path, "rb") as f: trend_models = pickle.load(f)

In [23]:
df_add_lists = []
for name in trend_models.keys():

    model_name = trend_models[name]
    df_pvalue  = model_name.pvalues.to_frame(name = "p_value").reset_index()
    df_tstat   = model_name.tvalues.to_frame(name = "t_stat").reset_index()
    df_coeff   = model_name.params.to_frame(name = "coeff").reset_index()

    df_add = (df_pvalue
        .merge(right = df_tstat, how = "inner", on = ["index"])
        .merge(right = df_coeff, how = "inner", on = ["index"])
        .assign(name = name, r2 = model_name.rsquared)
        .rename(columns = {"index": "param"}))

    df_add_lists.append(df_add)

df_trend_params = pd.concat(df_add_lists)

In [24]:
df_trend_param_prep = (df_trend_params
    .loc[lambda x: x.param == "signal"]
    .drop(columns = ["param"])
    .melt(id_vars = ["name"])
    .assign(
        str_split  = lambda x: x.name.str.split("-"),
        etf_ticker = lambda x: x.str_split.str[0],
        fx_ticker  = lambda x: x.str_split.str[1],
        vol_target = lambda x: x.str_split.str[2],
        signal_lag = lambda x: x.str_split.str[3])
    .drop(columns = ["name"]))

Now the next step is to run the same analysis with the z-scored signal with lagging to test its predictive capabilities. 

In [25]:
(df_trend_param_prep
    .assign(country = lambda x: x.fx_ticker.map(fx_country_mapper))
    .loc[lambda x: x.vol_target == "perfect"]
    .replace(RENAMER)
    .rename(columns = {
        "country"   : "Country",
        "etf_ticker": "ETF",
        "fx_ticker" : "FX",
        "signal_lag": "Signal",
        "variable"  : ""})
    .replace({
        "p_value": "$p$-value",
        "t_stat" : "$t$-stat",
        "coeff"  : "Coeff."})
    .pivot(index = ["Country", "ETF", "FX"], columns = ["", "Signal"], values = "value"))

$p$-value                 $t$-stat  \
Signal                               No Lag        Lagged     No Lag   
Country      ETF  FX                                                   
Argentina    ARGT ARS Carry    1.267131e-07  5.927038e-01  -5.293637   
Australia    EWA  AUD Carry    0.000000e+00  9.133858e-03 -50.854097   
                  AUD Future   0.000000e+00  1.109690e-01  50.933337   
Brazil       EWZ  BRL Carry    0.000000e+00  4.913769e-01 -64.451969   
                  BRL Future   0.000000e+00  1.452877e-08  60.282074   
Canada       EWC  CAD Carry   2.567761e-233  9.338415e-01 -33.798060   
                  CAD Future  1.164197e-295  9.356139e-01  38.470031   
Chile        ECH  CLP Carry   6.826668e-213  3.141903e-08 -32.852570   
China        MCHI CNY Carry    4.695994e-31  3.735820e-01 -11.693634   
                  CNY Future   6.267067e-27  1.620708e-07 -10.842152   
Colombia     COLO COP Carry   1.450555e-181  3.296224e-11 -30.168798   
Denmark      EDEN DKK Carry    1.850959e-90  2.296913e-02 -20.759949   
Euro         EZU  EUR Carry    0.000000e+00  1.903004e-01 -47.946535   
                  EUR Future   0.000000e+00  8.394959e-01  47.302195   
Hong Kong    EWH  HKD Carry    2.353982e-06  2.645291e-01  -4.724025   
Iceland      GLCR ISK Carry    6.780021e-07  8.307406e-01  -5.098009   
India        INDA INR Carry    1.233136e-93  7.122948e-06 -21.152386   
                  INR Future  5.658656e-115  4.349762e-03  23.732914   
Indonesia    EIDO IDR Carry    5.397158e-54  4.795345e-21 -15.707254   
Israel       EIS  ISR Future  4.878638e-134  2.979468e-01  25.501963   
Japan        EWJ  JPY Carry   1.477323e-122  1.826636e-02 -23.982462   
                  JPY Future  1.215344e-116  2.175317e-01  23.369181   
Korea        EWY  KRW Carry   1.654725e-228  1.932328e-29 -33.610773   
                  KRW Future   0.000000e+00  1.149524e-05  43.728939   
Malaysia     EWM  MYR Carry    1.968488e-19  1.712659e-01  -9.042153   
Mexico       EWW  MXN Carry    0.000000e+00  3.007790e-02 -43.064975   
                  MXN Future   0.000000e+00  6.892565e-06  45.642190   
New Zealand  ENZL NZD Carry   7.973881e-173  2.284571e-01 -29.494362   
                  NZD Future  3.247644e-192  4.445980e-01  31.333557   
Norway       ENOR NOK Carry   2.686294e-277  7.766503e-01 -38.992702   
                  NOK Future  2.282963e-286  4.258496e-01  39.813180   
Peru         EPU  PEN Carry    4.803474e-38  3.726872e-06 -13.023968   
Philippines  EPHE PHP Carry    1.745050e-33  1.309763e-10 -12.172926   
Poland       EPOL PLN Carry    0.000000e+00  1.880314e-02 -48.658629   
                  PLN Future   0.000000e+00  4.040121e-03  48.314278   
Singapore    EWS  SGD Carry   3.953310e-148  2.092684e-01 -26.569949   
South Africa EZA  ZAR Carry    0.000000e+00  1.933443e-02 -68.146493   
Sweden       EWD  SEK Carry    0.000000e+00  7.749530e-01 -51.410422   
                  SEK Future   0.000000e+00  2.440800e-02  51.131582   
Switzerland  EWL  CHF Carry    0.000000e+00  1.046636e-01 -44.012204   
                  CHF Future   0.000000e+00  8.528228e-01  42.946062   
Taiwan       EWT  TWD Carry   3.478130e-103  1.141035e-20 -21.963575   
Thailand     THD  THB Carry    2.834146e-82  7.744253e-18 -19.613610   
Turkey       TUR  TRY Carry    0.000000e+00  6.018265e-03 -43.205603   
UK           EWU  GBP Carry   1.048746e-308  1.174939e-01 -39.382759   
                  GBP Future  7.063060e-307  3.983069e-01  39.270683   

                                           Coeff.               $R^2$  \
Signal                           Lagged    No Lag    Lagged    No Lag   
Country      ETF  FX                                                    
Argentina    ARGT ARS Carry   -0.534967 -0.000576 -0.000058  0.007343   
Australia    EWA  AUD Carry    2.607676 -0.003450  0.000205  0.255040   
                  AUD Future  -1.594041  0.003421 -0.000124  0.257331   
Brazil       EWZ  BRL Carry   -0.688160 -0.004292 -0.000059  0.3912

In [50]:
df_tmp = (df_trend_param_prep
    .assign(group = lambda x: x.fx_ticker.map(fx_ticker_map))
    [["variable", "value", "vol_target", "signal_lag", "group", "fx_ticker"]]
    .loc[lambda x: x.vol_target != "lagged"])

In [76]:
df_sig_rate = (df_tmp
    .loc[lambda x: x.variable.isin(["p_value", "t_stat"])]
    .pivot(index = ["vol_target", "signal_lag", "group", "fx_ticker"], columns = "variable", values = "value")
    .assign(
        t_stat = lambda x: x.t_stat.abs(),
        stat_sig = lambda x: np.where((x.p_value < 0.05) & (x.t_stat > 1.2), "stat_sig", "not_sig"))
    .reset_index()
    [["vol_target", "signal_lag", "group", "stat_sig", "fx_ticker"]]
    .groupby(["vol_target", "signal_lag", "group", "stat_sig"])
    .agg("count")
    .reset_index()
    .pivot(index = ["vol_target", "signal_lag", "group"], columns = "stat_sig", values = "fx_ticker")
    .fillna(0)
    .assign(rate = lambda x: (x.stat_sig) / (x.not_sig + x.stat_sig))
    .reset_index()
    [["vol_target", "signal_lag", "group", "rate"]]
    .rename(columns = {"rate": "value"})
    .assign(variable = "sig_rate"))

In [77]:
df_tmp_avg = (df_tmp
    .drop(columns = ["fx_ticker"])
    .groupby(["variable", "vol_target", "signal_lag", "group"])
    .agg("mean")
    .reset_index())

In [94]:
display(pd
    .concat([df_sig_rate, df_tmp_avg])
    .replace(RENAMER)
    .replace({
        "sig_rate": "Stat Sig Rate",
        "p_value" : r"$p$-value",
        "t_stat"  : r"$t$-stat"})
    .rename(columns = {
        "group"     : "Group",
        "vol_target": "Vol. Target",
        "variable"  : "",
        "signal_lag": "Lag"})
    .pivot(index = ["Group", "Vol. Target"], columns = ["", "Lag"], values = "value"))

Stat Sig Rate               coeff            \
Lag                                 No Lag    Lagged    No Lag    Lagged   
Group            Vol. Target                                               
EM Carry         None                  1.0  0.652174 -0.002490 -0.000269   
                 Perf                  1.0  0.608696 -0.002385 -0.000336   
G10 Carry        None                  1.0  0.285714 -0.002978  0.000052   
                 Perf                  1.0  0.285714 -0.002954  0.000067   
Illiquid Futures None                  1.0  0.714286  0.002881  0.000163   
                 Perf                  1.0  0.714286  0.002692  0.000119   
Liquid Futures   None                  1.0  0.222222  0.003342  0.000133   
                 Perf                  1.0  0.222222  0.003044  0.000074   

                                  $p$-value               $R^2$            \
Lag                                  No Lag    Lagged    No Lag    Lagged   
Group            Vol. Target                                                
EM Carry         None          3.648458e-05  0.158779  0.140785  0.004735   
                 Perf          1.373347e-07  0.170013  0.149304  0.005069   
G10 Carry        None         4.267175e-126  0.289105  0.188583  0.000339   
                 Perf         2.110462e-123  0.324878  0.195886  0.000374   
Illiquid Futures None          1.705720e-26  0.126257  0.208836  0.002501   
                 Perf          8.952953e-28  0.108087  0.226215  0.002615   
Liquid Futures   None         4.206143e-124  0.388323  0.200641  0.001042   
                 Perf         1.350383e-117  0.422149  0.211132  0.000945   

                               $t$-stat            
Lag                              No Lag    Lagged  
Group            Vol. Target                       
EM Carry         None        -27.204387 -3.381636  
                 Perf        -28.104240 -3.392921  
G10 Carry        None        -36.978001  0.758566  
                 Perf        -37.778711  0.919790  
Illiquid Futures None         29.815195  1.112025  
                 Perf         31.625815  1.280097  
Liquid Futures   None         40.861338  1.111427  
                 Perf         42.172146  0.859552

In [26]:
df_avg_trend = (df_trend_param_prep
    .loc[lambda x: x.variable != "p_value"]
    .loc[lambda x: x.signal_lag == "no_lag"]
    .loc[lambda x: x.vol_target != "lagged"]
    .assign(
        value = lambda x: x.value.abs(),
        group = lambda x: x.fx_ticker.map(fx_ticker_map))
    [["variable", "value", "vol_target", "group"]]
    .groupby(["variable", "vol_target", "group"])
    .agg("mean")
    .reset_index()
    .assign(ols = "rtn_trend")
    .replace({
        "coeff": "param_val",
        "t_stat": "tvalue"}))

In [27]:
df_spread_avg = (df_spread_param_prep
    [["fx_ticker", "param_val", "tvalue", "r2", "vol_target"]]
    .assign(group = lambda x: x.fx_ticker.map(fx_ticker_map))
    .melt(id_vars = ["fx_ticker", "vol_target", "group"])
    .drop(columns = ["fx_ticker"])
    .loc[lambda x: x.vol_target != "lagged"]
    .groupby(["vol_target", "group", "variable"])
    .agg("mean")
    .reset_index()
    .assign(ols = "rtn_spread"))

In [28]:
df_rtn_avg = (df_rtn_beta
    .loc[lambda x: x.vol_target != "lagged"]
    .loc[lambda x: x.variable != "pvalue"]
    .assign(group = lambda x: x.fx_ticker.map(fx_ticker_map))
    [["vol_target", "variable", "value", "group"]]
    .groupby(["vol_target", "variable", "group"])
    .agg("mean")
    .reset_index()
    .assign(ols = "raw_rtn"))

In [29]:
exog_order = [
    "ETF Returns",
    "... Net SPY",
    "... Trend"
]

display(
    pd
    .concat([df_spread_avg, df_rtn_avg, df_avg_trend])
    .replace(RENAMER)
    .replace({
        "raw_rtn"   : "ETF Returns",
        "rtn_spread": "... Net SPY",
        "rtn_trend" : "... Trend"
    })
    .rename(columns={
        "group"     : "Group",
        "ols"       : "Exog",
        "vol_target": "Vol. Target",
        "variable"  : ""
    })
    .assign(
        Exog=lambda x: pd.Categorical(
            x["Exog"],
            categories=exog_order,
            ordered=True
        )
    )
    .pivot(
        index=["Group", "Exog"],
        columns=["Vol. Target", ""],
        values="value"
    )
)

Vol. Target                       None                           Perf  \
                                  Coef     $R^2$  $t$-value      Coef   
Group            Exog                                                   
EM Carry         ETF Returns  0.153894  0.207554  35.213062  0.144127   
                 ... Net SPY  0.175847  0.162720  29.658425  0.162772   
                 ... Trend    0.002490  0.140785  27.204387  0.002385   
G10 Carry        ETF Returns  0.224132  0.297788  48.423503  0.198262   
                 ... Net SPY  0.269947  0.200165  38.328634  0.252980   
                 ... Trend    0.002978  0.188583  36.978001  0.002954   
Illiquid Futures ETF Returns  0.191743  0.305372  38.823234  0.167442   
                 ... Net SPY  0.246241  0.251292  33.808184  0.212748   
                 ... Trend    0.003034  0.208836  32.832165  0.003054   
Liquid Futures   ETF Returns  0.203875  0.257971  46.774226  0.169663   
                 ... Net SPY  0.265264  0.211366  42.343619  0.233646   
                 ... Trend    0.003342  0.200641  40.861338  0.003044   

Vol. Target                                        
                                 $R^2$  $t$-value  
Group            Exog                              
EM Carry         ETF Returns  0.183813  31.954316  
                 ... Net SPY  0.149050  27.839508  
                 ... Trend    0.149304  28.104240  
G10 Carry        ETF Returns  0.246315  41.472738  
                 ... Net SPY  0.183244  36.004438  
                 ... Trend    0.195886  37.778711  
Illiquid Futures ETF Returns  0.266651  34.809710  
                 ... Net SPY  0.225437  31.164639  
                 ... Trend    0.226215  34.723573  
Liquid Futures   ETF Returns  0.220484  41.451710  
                 ... Net SPY  0.193828  39.861618  
                 ... Trend    0.211132  42.172146

In [30]:
(df_trend_param_prep)

,variable,value,str_split,etf_ticker,fx_ticker,vol_target,signal_lag
0,p_value,1.752388e-07,"[ARGT, USDARSCR Curncy, lagged, no_lag]",ARGT,USDARSCR Curncy,lagged,no_lag
1,p_value,2.993488e-01,"[ARGT, USDARSCR Curncy, lagged, with_lag]",ARGT,USDARSCR Curncy,lagged,with_lag
2,p_value,8.347260e-04,"[ARGT, USDARSCR Curncy, none, no_lag]",ARGT,USDARSCR Curncy,none,no_lag
3,p_value,6.445624e-01,"[ARGT, USDARSCR Curncy, none, with_lag]",ARGT,USDARSCR Curncy,none,with_lag
4,p_value,1.267131e-07,"[ARGT, USDARSCR Curncy, perfect, no_lag]",ARGT,USDARSCR Curncy,perfect,no_lag
...,...,...,...,...,...,...,...
1099,r2,1.403247e-03,"[TUR, USDTRYCR Curncy, lagged, with_lag]",TUR,USDTRYCR Curncy,lagged,with_lag
1100,r2,2.356688e-01,"[TUR, USDTRYCR Curncy, none, no_lag]",TUR,USDTRYCR Curncy,none,no_lag
1101,r2,1.256730e-03,"[TUR, USDTRYCR Curncy, none, with_lag]",TUR,USDTRYCR Curncy,none,with_lag
1102,r2,2.919619e-01,"[TUR, USDTRYCR Curncy, perfect, no_lag]",TUR,USDTRYCR Curncy,perfect,no_lag
